# Sentence Embeddings & Topic Modelling

Pipeline: Flan-T5 summarisation | MiniLM embeddings | FAISS vector store | BERTopic

In [7]:
import os, warnings
from pathlib import Path

import polars as pl
import numpy as np
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

CWD = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
NEWS_PATH = CWD / "data" / "Stock_news" / "subset_news.parquet"
MODEL_DIR = CWD / "models"
DATA_DIR = CWD / "data" / "Stock_news"
MODEL_DIR.mkdir(exist_ok=True)

# Paths for saved artefacts
SUMMARIES_PATH = DATA_DIR / "flan_summaries.parquet"
EMBEDDINGS_PATH = DATA_DIR / "chunk_embeddings.npy"
CHUNKS_PATH = DATA_DIR / "chunks.parquet"

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: mps


In [8]:
POOL = [
    'AAPL','MSFT','GOOG','GOOGL','AMZN','TSLA','META','NVDA','AMD','INTC','CRM','NFLX',
    'ADBE','PYPL','UBER','SQ','SHOP','ZM','SNAP','COIN','PLTR','ORCL',
    'QQQ','SPY','DIA','IWM',
    'T','VZ',
    'JPM','GS','MS','WFC','BAC','C',
    'XOM','CVX',
    'JNJ','PFE','MRNA','GILD','MRK','UNH','ABT',
    'WMT','COST','TGT','HD','KO','PEP','SBUX','MCD',
    'BA','GE','CAT','MMM',
    'DIS','CMCSA',
    'V','MA',
    'MU','QCOM','TXN','AVGO',
    'F','GM',
]
POOL_SET = set(POOL)

df = pl.read_parquet(NEWS_PATH)
df = df.filter(pl.col('Stock_symbol').is_in(POOL))
print(f'Full df: {df.shape[0]:,} rows')

Full df: 139,522 rows


In [9]:
# ---- SAMPLE SIZE (change this when running on full data) ----
N_SAMPLE = 1000
sample = df.sample(n=min(N_SAMPLE, df.shape[0]), seed=42)
print(f'Working sample: {sample.shape[0]} rows')
sample.head()

Working sample: 1000 rows


Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,date_parsed
str,str,str,str,str,str,str,str,str,str,str,date
"""2021-05-19 00:00:00 UTC""","""WMT August 20th Options Begin …","""WMT""","""https://www.nasdaq.com/article…",null,null,"""Investors in Walmart Inc (Symb…","""Of course, a lot of upside cou…","""Below is a chart showing WMT's…","""Below is a chart showing WMT's…","""At Stock Options Channel, our …",2021-05-19
"""2016-07-29 00:00:00 UTC""","""Emerging Asia & Utility: 2 ETF…","""QQQ""","""https://www.nasdaq.com/article…",null,null,"""In the last trading session, t…","""Among the top ETFs, investors …","""Click to get this free report …","""Click to get this free report …","""Click to get this free report …",2016-07-29
"""2019-11-25 00:00:00 UTC""","""Google's Weak Stadia Launch Di…","""GOOG""","""https://www.nasdaq.com/article…",null,null,"""In this episode of MarketFoole…","""You have five years and you've…","""Finally, Aaron and Emily disag…","""Finally, Aaron and Emily disag…","""Finally, Aaron and Emily disag…",2019-11-25
"""2023-08-03 00:00:00 UTC""","""3 ‘Strong Buy’ Dividend Stocks…","""PEP""","""https://www.nasdaq.com/article…",null,null,"""InvestorPlace - Stock Market N…","""PepsiCo recently raised the ea…","""PepsiCo (PEP) Source: FotograF…","""PepsiCo (PEP) Source: FotograF…","""PepsiCo (PEP) Source: FotograF…",2023-08-03
"""2013-09-06 00:00:00 UTC""","""Could Syria Really Sink the St…","""QQQ""","""https://www.nasdaq.com/article…",null,null,"""Submitted by Wall St. Daily as…","""Submitted by Wall St. Daily as…","""The views and opinions express…","""No matter how many times you r…","""No matter how many times you r…",2013-09-06


## 1. Summarisation with Flan-T5-small

Few-shot prompting with 2 examples. Compare against the existing `Lsa_summary` using ROUGE.

In [10]:
print("Article:", sample['Article'][0])
print("\nLsa_summary:", sample['Lsa_summary'][0])
print("-"*80)
print("Article:", sample['Article'][3])
print("\nLsa_summary:", sample['Lsa_summary'][3])


Article: Investors in Walmart Inc (Symbol: WMT) saw new options begin trading today, for the August 20th expiration. One of the key data points that goes into the price an option buyer is willing to pay, is the time value, so with 93 days until expiration the newly trading contracts represent a potential opportunity for sellers of puts or calls to achieve a higher premium than would be available for the contracts with a closer expiration. At Stock Options Channel, our YieldBoost formula has looked up and down the WMT options chain for the new August 20th contracts and identified one put and one call contract of particular interest.
The put contract at the $140.00 strike price has a current bid of $4.95. If an investor was to sell-to-open that put contract, they are committing to purchase the stock at $140.00, but will also collect the premium, putting the cost basis of the shares at $135.05 (before broker commissions). To an investor already interested in purchasing shares of WMT, that

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

FLAN_NAME = 'google/flan-t5-small'
FLAN_DIR = MODEL_DIR / 'flan-t5-small'

# Download once, then load from disk
if FLAN_DIR.exists():
    print(f'Loading Flan-T5 from {FLAN_DIR}')
    flan_tok = AutoTokenizer.from_pretrained(FLAN_DIR)
    flan_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_DIR)
else:
    print(f'Downloading {FLAN_NAME}...')
    flan_tok = AutoTokenizer.from_pretrained(FLAN_NAME)
    flan_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_NAME)
    flan_tok.save_pretrained(FLAN_DIR)
    flan_model.save_pretrained(FLAN_DIR)
    print(f'Saved to {FLAN_DIR}')

flan_model = flan_model.to(DEVICE).eval()
print('Flan-T5 ready')

Loading Flan-T5 from /Users/armandhubler/Documents/coding_project/nlp_project/models/flan-t5-small
Flan-T5 ready


In [12]:
# Two few-shot examples baked into the prompt
FEW_SHOT_PROMPT = """Summarize the following news article in 2-3 sentences.

Example 1:
Article: Apple reported record quarterly revenue of $123.9 billion, up 11% year over year. \
iPhone revenue grew 9% to $71.4 billion. CEO Tim Cook said the company saw strong demand \
across all product categories and geographic segments.
Summary: Apple achieved record Q1 revenue of $123.9B driven by 9% iPhone growth. \
CEO Cook highlighted broad strength across products and regions.

Example 2:
Article: Tesla delivered 405,278 vehicles in Q4, missing analyst expectations of 420,760. \
The company cited logistical challenges and year-end demand shifts. Full-year deliveries \
reached 1.31 million, a 40% increase over the prior year.
Summary: Tesla Q4 deliveries of 405K missed estimates due to logistics issues. \
Full-year deliveries hit 1.31M, up 40% YoY.

Article: {article}
Summary:"""


def summarise_article(article, max_input=512, max_output=80):
    """Generate a summary using Flan-T5 with few-shot prompt."""
    prompt = FEW_SHOT_PROMPT.format(article=article[:1500])
    inputs = flan_tok(prompt, return_tensors='pt', max_length=max_input, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = flan_model.generate(**inputs, max_new_tokens=max_output, num_beams=2, early_stopping=True)
    return flan_tok.decode(out[0], skip_special_tokens=True)

In [13]:
articles = sample['Article'].to_list()
lsa_summaries = sample['Lsa_summary'].to_list()

In [14]:
if SUMMARIES_PATH.exists():
    print(f"Loading cached summaries from {SUMMARIES_PATH}")
    _sum_df = pl.read_parquet(SUMMARIES_PATH)
    flan_summaries = _sum_df["flan_summary"].to_list()
else:
    flan_summaries = []
    for art in tqdm(articles, desc="Summarising"):
        flan_summaries.append(summarise_article(art))

    # Save summaries to data dir
    _sum_df = pl.DataFrame({"flan_summary": flan_summaries})
    _sum_df.write_parquet(SUMMARIES_PATH)
    print(f"Saved {len(flan_summaries)} summaries to {SUMMARIES_PATH}")

# Show first result
print("Article (first 300 chars):", articles[0][:300])
print("\nFlan-T5 summary:", flan_summaries[0])
print("\nLSA summary (first 200 chars):", lsa_summaries[0][:200])

Summarising:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved 1000 summaries to /Users/armandhubler/Documents/coding_project/nlp_project/data/Stock_news/flan_summaries.parquet
Article (first 300 chars): Investors in Walmart Inc (Symbol: WMT) saw new options begin trading today, for the August 20th expiration. One of the key data points that goes into the price an option buyer is willing to pay, is the time value, so with 93 days until expiration the newly trading contracts represent a potential opp

Flan-T5 summary: Stock Options Channel will track the odds of the new contracts for August 20th.

LSA summary (first 200 chars): Of course, a lot of upside could potentially be left on the table if WMT shares really soar, which is why looking at the trailing twelve month trading history for Walmart Inc, as well as studying the 


In [15]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Score Flan-T5 summaries against the full article (as reference)
flan_scores = [scorer.score(art, s) for art, s in zip(articles, flan_summaries)]
lsa_scores  = [scorer.score(art, s) for art, s in zip(articles, lsa_summaries)]

def avg_rouge(scores):
    """Average ROUGE F1 across samples."""
    return {
        k: np.mean([s[k].fmeasure for s in scores])
        for k in ['rouge1', 'rouge2', 'rougeL']
    }

print('Flan-T5 ROUGE:', {k: f'{v:.3f}' for k, v in avg_rouge(flan_scores).items()})
print('LSA    ROUGE:', {k: f'{v:.3f}' for k, v in avg_rouge(lsa_scores).items()})

Flan-T5 ROUGE: {'rouge1': '0.072', 'rouge2': '0.060', 'rougeL': '0.068'}
LSA    ROUGE: {'rouge1': '0.289', 'rouge2': '0.282', 'rougeL': '0.256'}


## 2. Sentence Embeddings & FAISS

Embed chunked summaries with `all-MiniLM-L6-v2`, store in a FAISS index.

In [16]:
from sentence_transformers import SentenceTransformer

EMBED_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIR = MODEL_DIR / 'all-MiniLM-L6-v2'

if EMBED_DIR.exists():
    print(f'Loading embedding model from {EMBED_DIR}')
    embed_model = SentenceTransformer(str(EMBED_DIR), device=DEVICE)
else:
    print(f'Downloading {EMBED_NAME}...')
    embed_model = SentenceTransformer(EMBED_NAME, device=DEVICE)
    embed_model.save(str(EMBED_DIR))
    print(f'Saved to {EMBED_DIR}')

print('Embedding model ready')

Loading embedding model from /Users/armandhubler/Documents/coding_project/nlp_project/models/all-MiniLM-L6-v2
Embedding model ready


In [17]:
titles  = sample['Article_title'].to_list()
dates   = sample['Date'].to_list()
symbols = sample['Stock_symbol'].to_list()


def make_chunks(summary, title, date, symbol):
    """Split a summary into 2 chunks, each prefixed with metadata."""
    sentences = summary.split('. ')
    mid = max(1, len(sentences) // 2)
    chunk1 = '. '.join(sentences[:mid]) + '.'
    chunk2 = '. '.join(sentences[mid:]).strip()
    if not chunk2:
        chunk2 = chunk1  # fallback for single-sentence summaries

    prefix = f'[{symbol}] [{date}] {title} | '
    return [prefix + chunk1, prefix + chunk2]


# Build chunks from Flan-T5 summaries
chunks, chunk_meta = [], []
for i in range(len(flan_summaries)):
    for chunk in make_chunks(flan_summaries[i], titles[i], dates[i], symbols[i]):
        chunks.append(chunk)
        chunk_meta.append({'idx': i, 'symbol': symbols[i], 'date': dates[i], 'title': titles[i]})

print(f'{len(chunks)} chunks from {len(flan_summaries)} articles')
print('Sample chunk:', chunks[0][:200])

2000 chunks from 1000 articles
Sample chunk: [WMT] [2021-05-19 00:00:00 UTC] WMT August 20th Options Begin Trading | Stock Options Channel will track the odds of the new contracts for August 20th..


In [18]:
import faiss

if EMBEDDINGS_PATH.exists() and CHUNKS_PATH.exists():
    print(f"Loading cached embeddings from {EMBEDDINGS_PATH}")
    embeddings = np.load(EMBEDDINGS_PATH)
    _ch_df = pl.read_parquet(CHUNKS_PATH)
    chunks = _ch_df["chunk"].to_list()
    chunk_meta = _ch_df.drop("chunk").to_dicts()
else:
    # Encode all chunks
    embeddings = embed_model.encode(chunks, show_progress_bar=True, batch_size=32)
    embeddings = embeddings.astype("float32")

    # Save embeddings and chunks to data dir
    np.save(EMBEDDINGS_PATH, embeddings)
    _ch_df = pl.DataFrame([{"chunk": c, **m} for c, m in zip(chunks, chunk_meta)])
    _ch_df.write_parquet(CHUNKS_PATH)
    print(f"Saved embeddings {embeddings.shape} to {EMBEDDINGS_PATH}")

embeddings = embeddings.astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"FAISS index size: {index.ntotal}")

# Save index
FAISS_PATH = MODEL_DIR / "news_embeddings.faiss"
faiss.write_index(index, str(FAISS_PATH))
print(f"Saved FAISS index to {FAISS_PATH}")

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Saved embeddings (2000, 384) to /Users/armandhubler/Documents/coding_project/nlp_project/data/Stock_news/chunk_embeddings.npy
Embeddings shape: (2000, 384)
FAISS index size: 2000
Saved FAISS index to /Users/armandhubler/Documents/coding_project/nlp_project/models/news_embeddings.faiss


In [19]:
# Quick retrieval test
query = 'tech company earnings report'
q_vec = embed_model.encode([query]).astype('float32')
D, I = index.search(q_vec, k=3)

print(f'Query: "{query}"\n')
for rank, (dist, idx) in enumerate(zip(D[0], I[0])):
    print(f'  #{rank+1} (dist={dist:.2f}): {chunks[idx][:150]}')

Query: "tech company earnings report"

  #1 (dist=0.76): [TXN] [2017-10-24 00:00:00 UTC] Texas Instruments Beats Third-Quarter Sales, Earnings Views | Dallas-based Texas Instruments posted earnings per share
  #2 (dist=0.82): [TSLA] [2022-10-19 00:00:00 UTC] After-Hours Earnings Report for October 19, 2022 : TSLA, IBM, CCI, LRCX, KMI, PPG, LVS, EFX, STLD, REXR, KNX, AA | Su
  #3 (dist=0.82): [TSLA] [2022-10-19 00:00:00 UTC] After-Hours Earnings Report for October 19, 2022 : TSLA, IBM, CCI, LRCX, KMI, PPG, LVS, EFX, STLD, REXR, KNX, AA | Su


## 3. BERTopic — Topic Modelling

Cluster the embedded chunks into topics using UMAP + HDBSCAN.

In [20]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

n_docs = len(chunks)

# Scale params to dataset size (small sample vs full run)
umap_neighbors = min(15, n_docs - 1)
umap_components = min(5, n_docs - 2)
hdbscan_min_cluster = max(2, min(50, n_docs // 5))

umap_model = UMAP(
    n_neighbors=umap_neighbors,
    n_components=umap_components,
    min_dist=0.0,
    metric='cosine',
    low_memory=True,
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=hdbscan_min_cluster,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# Strip metadata noise from c-TF-IDF representation
EXTRA_STOPS = [
    'utc', '00', 'summary', 'stock', 'stocks', 'inc', 'corp', 'company',
    'nasdaq', 'shares', 'share', 'said', 'also', 'new', 'would', 'could',
    *[t.lower() for t in POOL],  # ticker symbols are not topical
]

vectorizer = CountVectorizer(
    stop_words='english',
    min_df=3,
    ngram_range=(1, 2),
)
# Extend sklearn's english stop words with our custom list
vectorizer.stop_words = list(
    set(vectorizer.get_stop_words()) | set(EXTRA_STOPS)
)

topic_model = BERTopic(
    embedding_model=embed_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    calculate_probabilities=False,
    verbose=True
)

print(f'BERTopic config: {n_docs} docs, umap_neighbors={umap_neighbors}, '
      f'umap_components={umap_components}, hdbscan_min_cluster={hdbscan_min_cluster}')

# Fit on pre-computed embeddings
topics, probs = topic_model.fit_transform(chunks, embeddings=embeddings)
print(f'Found {len(set(topics)) - (1 if -1 in topics else 0)} topics (+ outlier cluster)')

2026-04-11 16:01:48,772 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


BERTopic config: 2000 docs, umap_neighbors=15, umap_components=5, hdbscan_min_cluster=50


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2026-04-11 16:01:57,375 - BERTopic - Dimensionality - Completed ✓
2026-04-11 16:01:57,375 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-11 16:01:57,428 - BERTopic - Cluster - Completed ✓
2026-04-11 16:01:57,433 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-11 16:01:57,482 - BERTopic - Representation - Completed ✓


Found 8 topics (+ outlier cluster)


In [21]:
# Topic overview
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

   Topic  Count                                 Name  \
0     -1    623         -1_2023_earnings_2022_market   
1      0    578       0_wall_wall street_2023_street   
2      1    182     1_earnings_quarter_mobile_report   
3      2    167           2_etf_week_symbol_increase   
4      3    126           3_covid_19_covid 19_health   
5      4    110       4_walmart_target_consumer_2020   
6      5     78                5_oil_energy_gas_2022   
7      6     78  6_dividend_date_begin trading_begin   
8      7     58       7_intel_apple_citigroup_israel   

                                      Representation  \
0  [2023, earnings, 2022, market, 11, 10, 2018, p...   
1  [wall, wall street, 2023, street, market, buy,...   
2  [earnings, quarter, mobile, report, q3, zacks,...   
3  [etf, week, symbol, increase, 000, holdings, 5...   
4  [covid, 19, covid 19, health, merck, 2020, ana...   
5  [walmart, target, consumer, 2020, 2023, 2021, ...   
6  [oil, energy, gas, 2022, sector, talks, 2020

In [22]:
# Show top words for each topic (skip outlier -1)
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)[:5]]
    print(f'Topic {tid}: {" | ".join(words)}')

Topic 0: wall | wall street | 2023 | street | market
Topic 1: earnings | quarter | mobile | report | q3
Topic 2: etf | week | symbol | increase | 000
Topic 3: covid | 19 | covid 19 | health | merck
Topic 4: walmart | target | consumer | 2020 | 2023
Topic 5: oil | energy | gas | 2022 | sector
Topic 6: dividend | date | begin trading | begin | 2018
Topic 7: intel | apple | citigroup | israel | business


In [23]:
# Save topic model
TOPIC_DIR = MODEL_DIR / 'bertopic_news'
topic_model.save(str(TOPIC_DIR), serialization='safetensors', save_ctfidf=True, save_embedding_model=EMBED_DIR)
print(f'Saved BERTopic model to {TOPIC_DIR}')

Saved BERTopic model to /Users/armandhubler/Documents/coding_project/nlp_project/models/bertopic_news
